In [1]:
import os
from dotenv import load_dotenv
load_dotenv("/home/abhi/AI_Workspace/personal/Generative-AI-Engineer-Portfolio/.env")


True

In [2]:
from langchain_ollama import OllamaLLM
llm = "phi3:latest"
chat_model = OllamaLLM(model=llm, temperature=0.7, max_tokens=2048)
chat_model

OllamaLLM(model='phi3:latest', temperature=0.7)

In [3]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

In [4]:
speech = """
My colleagues in the Union Cabinet, Ram Mohan Naidu Ji, Murlidhar Mohol Ji, Honorable Ministers from across the world, leaders of the global aviation industry, representatives of international organizations, Distinguished Delegates, ladies and gentlemen!

​Greetings!

​On this platform of Wings India, I welcome all industry leaders, experts, and investors. As you all know, the next era of the aviation industry is filled with many aspirations. And India is becoming a major player in this. Aircraft Manufacturing, Pilot Training, Advanced Air Mobility, Aircraft Leasing – these are sectors with which India stands before you with numerous possibilities. And therefore, this summit of Wings India has become so significant for all of us.

​Friends,

​In the last decade, a historic transformation has taken place in India’s entire aviation sector. There was a time when India was among those countries where air travel was limited to an exclusive club. But today, the situation has completely changed. Today, India is the world’s third-largest domestic aviation market. Our passenger traffic has increased very rapidly. The fleet of Indian airlines is also expanding fast. In recent years, India’s airlines have placed orders for more than 1500 airplanes.
"""

In [5]:
chat_message = [
    SystemMessage(content="You are an expert with expertise in summarizing the speeches."),
    HumanMessage(content=f"Provide a short and concise summary of the speech:\n \
                 Speech Text: {speech}"),
]

In [6]:
chat_model.get_num_tokens(speech)

262

In [7]:
chat_model.invoke(chat_message)

"The speaker addresses industry leaders and delegates at a Wings India summit to discuss the significant transformation in India's aviation sector over the past decade. He highlights how from being a country with restricted domestic air travel, it has now become the world’s third-largest market due to rapid growth in passenger traffic, fleet expansion of Indian Airlines and substantial aircraft orders exceeding 1500 planes recently."

In [16]:
##Prompt template text summarization
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

generic_template = """
Generate a translated short summary of the following speech in the language given for translation.
Tranlsate to: {language}
Speech: {speech}
"""

prompt = PromptTemplate(
    input_variables=["speech", "language"],
    template=generic_template,
    output_parser=StrOutputParser()
)

prompt

PromptTemplate(input_variables=['language', 'speech'], input_types={}, output_parser=StrOutputParser(), partial_variables={}, template='\nGenerate a translated short summary of the following speech in the language given for translation.\nTranlsate to: {language}\nSpeech: {speech}\n')

In [15]:
complete_prompt = prompt.format(speech=speech, language="Spanish")
complete_prompt

'\nGenerate a translated short summary of the following speech in the language given for translation.\nTranlsate to: Spanish\nSpeech: \nMy colleagues in the Union Cabinet, Ram Mohan Naidu Ji, Murlidhar Mohol Ji, Honorable Ministers from across the world, leaders of the global aviation industry, representatives of international organizations, Distinguished Delegates, ladies and gentlemen!\n\n\u200bGreetings!\n\n\u200bOn this platform of Wings India, I welcome all industry leaders, experts, and investors. As you all know, the next era of the aviation industry is filled with many aspirations. And India is becoming a major player in this. Aircraft Manufacturing, Pilot Training, Advanced Air Mobility, Aircraft Leasing – these are sectors with which India stands before you with numerous possibilities. And therefore, this summit of Wings India has become so significant for all of us.\n\n\u200bFriends,\n\n\u200bIn the last decade, a historic transformation has taken place in India’s entire avi

In [10]:
chat_model.get_num_tokens(complete_prompt)

293

In [11]:
chain = prompt | chat_model | StrOutputParser()
response = chain.invoke({"speech": speech, "language": "Spanish"})
response

'Distinguidos compañeros en el Gabinete Unido, Ram Mohan Naidu Ji, Murlidhar Mohol Ji, distinguidos Ministros de todo el mundo, líderes de la industria global del transporte aéreo, representantes de organizaciones internacionales y Distintos Embajadores!\n\n¡Saludos!\n\nEn este plataforma de Wings India, les saludo a todos los líderes de la industria, expertos e inversionistas. Como saben ustedes, el próximo período del sector de transporte aéreo está lleno de aspiraciones. Y en nuestro país es que estamos emergiendo como un jugador importante en este campo. La fabricación de aviones, la formación de pilotos, el movimiento aéreo avanzado y la alquiler de aviones – estos son sectores con los cuales India se presenta ante ustedes llenos de posibilidades. Por lo tanto, esta cumbre del Wings India ha adquirido una gran importancia para todos nosotros.\n\nAmigos,\n\nEn la última década, ha ocurrido un cambio histórico en todo el sector de aviación de nuestro país. Hace tiempo que en este lu

In [19]:
#StuffDocumentChain for text summarization
"""
If using old version ( < 1.0) of langchain. We can use the following code. But in the latest version of langchain ( >= 1.0), we can use Runnables and LCEL as shown in the next code cells.
from langchain.chains.summarize import load_summarize_chain

chain = load_summarize_chain(llm, chain_type="stuff")
result = chain.invoke(docs)
"""

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("LangChain Runnable & LCEL.pdf")
docs = loader.load_and_split()
docs

[Document(metadata={'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'creationdate': '2026-01-07T08:57:26+00:00', 'moddate': '2026-01-07T08:57:26+00:00', 'source': 'LangChain Runnable & LCEL.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='1. What is a Runnable?\nRunnable is the core abstraction in modern LangChain.\nDe\x00nition A Runnable is a building-block unit of work that:\ntakes an input\ntransforms it into an output\ncan be composed with other runnables\nsupports batching, async, streaming, tracing, retries, etc.\nA Runnable implements a standard interface with methods like:\ninvoke() — run once\n.batch() / .abatch() — run in parallel\n.stream() / .astream() — stream output as it’s generated\nchaining support using the pipe operator ( |) ([LangChain][1])\nExamples of Runnables include:\nPrompt templates\nLLM / chat model wrappers\nOutput parsers\nRetrievers\nCustom logic wrapped as a RunnableLambda ([Medium][2])\nKey point: Each of these implements the Run

In [20]:
from langchain_openai import AzureChatOpenAI

chat_model = AzureChatOpenAI(
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
            azure_deployment='gpt-5.2-chat',
            api_version="2025-04-01-preview",
            api_key=os.getenv("AZURE_OPENAI_API_KEY"),
)

chat_model

AzureChatOpenAI(profile={}, client=<openai.resources.chat.completions.completions.Completions object at 0x71c8610382d0>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x71c861039350>, root_client=<openai.lib.azure.AzureOpenAI object at 0x71c861052010>, root_async_client=<openai.lib.azure.AsyncAzureOpenAI object at 0x71c861076a50>, model_kwargs={}, openai_api_key=SecretStr('**********'), stream_usage=True, disabled_params={'parallel_tool_calls': None}, azure_endpoint='https://pfs-2-namit-resource.cognitiveservices.azure.com/', deployment_name='gpt-5.2-chat', openai_api_version='2025-04-01-preview', openai_api_type='azure')

In [21]:
template = """ Write a concise summary of the following text:
<text>
{text}
</text>
"""

prompt = PromptTemplate(input_variables=["text"], template=template)
prompt

PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template=' Write a concise summary of the following text:\n<text>\n{text}\n</text>\n')

In [22]:
from langchain_core.runnables import RunnableSequence
summarize_runnable = ( prompt | chat_model | StrOutputParser() )

def summarize_docs(docs: list) -> str:
    combined = "\n\n".join(doc.page_content for doc in docs)
    return summarize_runnable.invoke({"text": combined})

In [23]:
summary = summarize_docs(docs)
summary

'LangChain v1.x replaces the old `langchain.chains` API with **Runnables** and the **LangChain Expression Language (LCEL)**. A **Runnable** is the core abstraction: a composable unit of work that transforms input to output and supports sync/async execution, batching, streaming, retries, and tracing. Prompts, LLMs, parsers, retrievers, and custom logic all implement the Runnable interface.\n\n**LCEL** is a declarative syntax for composing Runnables into pipelines using operators like `|`, enabling clear, modular, and optimized workflows, including parallel execution via `RunnableParallel`. Together, Runnables and LCEL fully replace legacy chains, providing a cleaner, more flexible, and more powerful way to build LangChain pipelines.'

In [24]:
#MapReduce for text summarization of large documents
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)

split_docs = text_splitter.split_documents(docs)
split_docs

[Document(metadata={'producer': 'Skia/PDF m97', 'creator': 'Chromium', 'creationdate': '2026-01-07T08:57:26+00:00', 'moddate': '2026-01-07T08:57:26+00:00', 'source': 'LangChain Runnable & LCEL.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1'}, page_content='1. What is a Runnable?\nRunnable is the core abstraction in modern LangChain.\nDe\x00nition A Runnable is a building-block unit of work that:\ntakes an input\ntransforms it into an output\ncan be composed with other runnables\nsupports batching, async, streaming, tracing, retries, etc.\nA Runnable implements a standard interface with methods like:\ninvoke() — run once\n.batch() / .abatch() — run in parallel\n.stream() / .astream() — stream output as it’s generated\nchaining support using the pipe operator ( |) ([LangChain][1])\nExamples of Runnables include:\nPrompt templates\nLLM / chat model wrappers\nOutput parsers\nRetrievers\nCustom logic wrapped as a RunnableLambda ([Medium][2])\nKey point: Each of these implements the Run

In [25]:
chunks_prompt = """
Summarize the piece of text below from the document:
Text: {text}
Summary:
"""

map_prompt_template = PromptTemplate(
    input_variables=["text"],
    template=chunks_prompt,
)

In [27]:
final_prompt = """
Provide the final summary of the entire text with these important points.
Add a suitable title, start the precise summary with an introduction and provide the summary in bullet points.
<Points>
{summaries}
</Points>
"""

reduce_prompt_template = PromptTemplate(
    input_variables=["summaries"],
    template=final_prompt,
)

In [30]:
"""
If using old version ( < 1.0) of langchain. We can use the following code. But in the latest version of langchain ( >= 1.0), we can use Runnables and LCEL as shown in the next code cells.

from langchain.chains.summarize import load_summarize_chain

summary_chain = load_summarize_chain(llm, chain_type="map_reduce", map_prompt=map_prompt_template, combine_prompt=reduce_prompt_template)
result = chain.run(split_docs)
"""

map_runnable = map_prompt_template | chat_model | StrOutputParser()
reduce_runnable = reduce_prompt_template | chat_model | StrOutputParser()

def map_reduce_from_split_docs(split_docs: list):
    # → Map step — summarize each chunk
    intermediate_summaries = []
    
    for doc in split_docs:
        text = doc.page_content
        partial_summary = map_runnable.invoke({"text": text})
        intermediate_summaries.append(partial_summary)

    # → Reduce step — combine intermediate summaries
    joined_summaries = "\n\n".join(intermediate_summaries)
    final_summary = reduce_runnable.invoke({"summaries": joined_summaries})
    
    return final_summary

In [31]:
summary = map_reduce_from_split_docs(split_docs)
print(summary)

## **LangChain v1.x+: Runnables and LCEL Explained**

### **Introduction**
The text describes LangChain’s architectural shift in v1.x+ toward a more modular, declarative, and powerful system built around **Runnables** and the **LangChain Expression Language (LCEL)**. This new approach replaces older chain-based APIs with flexible pipelines that are easier to compose, optimize, and scale.

### **Final Summary (Key Points)**

- **Runnables as the Core Abstraction**
  - A **Runnable** is the fundamental building block in modern LangChain.
  - It represents a unit of work that transforms input into output.
  - All Runnables share a standard interface (`invoke`, `batch`, `stream`, `ainvoke`, etc.).
  - Examples include prompt templates, LLM/chat model wrappers, retrievers, output parsers, and custom logic.
  - Runnables support advanced features such as async execution, batching, streaming, retries, tracing, and parallelism.

- **LCEL (LangChain Expression Language)**
  - LCEL is a declarat

In [32]:
#RefineChain for text summarization
"""
If using old version ( < 1.0) of langchain. We can use the following code. But in the latest version of langchain ( >= 1.0), we can use Runnables and LCEL as shown in the next code cells.

from langchain.chains.summarize import load_summarize_chain

summary_chain = load_summarize_chain(llm, chain_type="refine")
result = chain.run(split_docs)
"""

from langchain_core.runnables import RunnableSequence

initial_prompt = PromptTemplate.from_template(
    """Here is a document chunk:

{chunk_text}

Produce a standalone summary of this chunk:
"""
)

refine_prompt = PromptTemplate.from_template(
    """We have an existing summary up to now:

{existing_summary}

Now consider this new document chunk:

{chunk_text}

Refine the summary only if it adds useful information.
If the new context does *not* change meaning, return the original summary.
"""
)

In [33]:
initial_runnable = (
    initial_prompt
    | chat_model
    | StrOutputParser()
)

refine_runnable = (
    refine_prompt
    | chat_model
    | StrOutputParser()
)

In [36]:
from langchain_core.callbacks import CallbackManager, StdOutCallbackHandler

callback_mgr = CallbackManager([StdOutCallbackHandler()])

def refine_summarize_from_split_docs(split_docs: list):
    """
    LangChain v1 replacement for old chain_type='refine' summarization.
    split_docs should be a list of Document objects already split using a text splitter.
    """

    # 1) Start with first chunk
    first_chunk = split_docs[0].page_content
    current_summary = initial_runnable.invoke({"chunk_text": first_chunk}, config={"callbacks": callback_mgr})

    # 2) Iteratively refine with each subsequent chunk
    for doc in split_docs[1:]:
        new_chunk_text = doc.page_content
        current_summary = refine_runnable.invoke({
            "existing_summary": current_summary,
            "chunk_text": new_chunk_text
        }, config={"callbacks": callback_mgr})

    return current_summary

In [37]:
summary = refine_summarize_from_split_docs(split_docs)
print(summary)



> Entering new RunnableSequence chain...

> Finished chain.


> Entering new RunnableSequence chain...

> Finished chain.


> Entering new RunnableSequence chain...

> Finished chain.


> Entering new RunnableSequence chain...

> Finished chain.


> Entering new RunnableSequence chain...

> Finished chain.


> Entering new RunnableSequence chain...

> Finished chain.


> Entering new RunnableSequence chain...

> Finished chain.
---

### **Refined Summary**

A **Runnable** is the core abstraction in modern LangChain, representing a reusable unit of work that takes an input, transforms it into an output, and can be composed with other Runnables. All Runnables share a standard interface that supports single execution (`invoke`), batching (`batch`/`abatch`), streaming (`stream`/`astream`), async execution, retries, and tracing. Common examples include prompt templates, LLM or chat model wrappers, output parsers, retrievers, and custom logic wrapped as `RunnableLambda`. Because they all i

In [38]:
from langchain_core.callbacks.base import BaseCallbackHandler

class ShowPromptOutputHandler(BaseCallbackHandler):
    def on_chat_model_start(self, serialized, messages, **kwargs):
        print("PROMPT:", messages)
    def on_llm_end(self, response, **kwargs):
        print("RESPONSE:", response.output_text)

callback_mgr = CallbackManager([ShowPromptOutputHandler()])

def refine_summarize_from_split_docs(split_docs: list):
    """
    LangChain v1 replacement for old chain_type='refine' summarization.
    split_docs should be a list of Document objects already split using a text splitter.
    """

    # 1) Start with first chunk
    first_chunk = split_docs[0].page_content
    current_summary = initial_runnable.invoke({"chunk_text": first_chunk}, config={"callbacks": callback_mgr})

    # 2) Iteratively refine with each subsequent chunk
    for doc in split_docs[1:]:
        new_chunk_text = doc.page_content
        current_summary = refine_runnable.invoke({
            "existing_summary": current_summary,
            "chunk_text": new_chunk_text
        }, config={"callbacks": callback_mgr})

    return current_summary

summary = refine_summarize_from_split_docs(split_docs)
print(summary)

### Refined Summary (Updated with Comparison Context)

In LangChain, a **Runnable** is the core abstraction representing a unit of work that takes an input, transforms it into an output, and can be composed with other Runnables. Runnables share a standard interface that supports single execution (`invoke`), batching, asynchronous execution, streaming (`stream()` / `astream()`), retries, and tracing. Common examples include prompt templates, LLM or chat model wrappers, output parsers, retrievers, and custom logic wrapped as `RunnableLambda`. Because they all implement the same interface, they can be seamlessly combined.

**LCEL (LangChain Expression Language)** is a declarative syntax for composing Runnables into pipelines using operators such as the pipe (`|`). For example:

```python
pipeline = prompt | llm | parser
result = pipeline.invoke({"input": ...})
```

This works because each component is a Runnable. LCEL **replaces the older `langchain.chains` API** and shifts LangChain towa